# Monte Carlo Execution Showcase (IEOR4703)

Applies the course's Monte Carlo toolkit to the regime-conditioned limit-order strategy.
Each regime's range distribution is defined **three ways and compared**: empirical (benchmark), a fitted parametric family (AIC-selected), and a Brownian-motion path model.

Reusable logic lives in `src/order_mgmt/mc/`; this notebook only orchestrates.
Reproducible: a single seed is set below.

In [ ]:
import sys, math
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

SEED = 0
ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
FIGDIR = ROOT / "reports" / "figures"; FIGDIR.mkdir(parents=True, exist_ok=True)

from plot_volume import _compute_stats
from ranges import compute_all_ranges
from regime import compute_ewma_series
from epdf import build_epdf
from order_mgmt.pipeline import load_market_indexed
from order_mgmt.ticks import resolve_tick
from order_mgmt.backtest import run_backtest_rolling
from order_mgmt.mc.bootstrap import bootstrap_strategy
from order_mgmt.mc.samplers import sample_from_epdf
from order_mgmt.mc.fit import fit_distribution, fit_all_regimes
from order_mgmt.mc.paths import simulate_gbm_ranges, halfnormal_range_mean, sigma_from_range_level
from order_mgmt.mc.simulator import run_marginal_mc, stress_sigma
from order_mgmt.mc.variance_reduction import (
    antithetic_slippage, control_variate_slippage,
    conditional_mc_fill_rate, importance_sampling_chase_tail,
)
from order_mgmt.mc.validation import compare_mc_vs_backtest, compare_models

TAU, HALF_LIFE, M, N, K, J_START, TARGET, SIDE = 5, 20, 3, 3, 3, 200, 0.6, "sell"

## Setup — load Gold, build per-regime ePDFs, fits, σ, weights

In [ ]:
mkt = next(p for p in (ROOT / "data").iterdir() if "gold" in p.name.lower())
df = load_market_indexed(mkt)
ohlcv = df[["open","high","low","close","volume"]]
_, inferred, days, _, _ = _compute_stats(ohlcv)
tick = resolve_tick(df["contract"].iloc[0], inferred)

t_list, ell_r, ell_u, ell_d, vol_list, dx_list = compute_all_ranges(ohlcv, TAU, tick, days)
ewma_range, ewma_vol = compute_ewma_series(t_list, ell_r, vol_list, HALF_LIFE)
counts_RU, counts_RD, _ = build_epdf(t_list, ell_u, ell_d, list(ewma_vol), list(ewma_range),
                                     dx_list, M=M, N=N, K=K, j_start=J_START)

def cell_mean(c):
    tot = sum(c.values()); return sum(l*n for l,n in c.items())/tot if tot else 0.0
decision = counts_RU                                   # sell -> rangeUp
cell_weights = {c: sum(decision[c].values()) for c in decision}
sigma_by_cell = {c: sigma_from_range_level(cell_mean(counts_RU[c])+cell_mean(counts_RD[c]), TAU)
                 for c in decision}
fits = fit_all_regimes(decision)
busiest = max(decision, key=lambda c: sum(decision[c].values()))
print(f"{mkt.name}: tick={tick:g}, {len(decision)} regimes; busiest cell {busiest} "
      f"with {sum(decision[busiest].values())} obs")

## Phase A — Bootstrap confidence intervals
Treat the regime's rangeUp `Counter` as the empirical CDF F̃ and bootstrap the fill rate at ℓ* (cheat-sheet Bootstrap Algorithm + basic CI).

In [ ]:
bs = bootstrap_strategy(decision[busiest], side=SIDE, fill_rate_target=TARGET, n_boot=2000, seed=SEED)
print(f"cell {busiest}: ell*={bs.ell_star}")
print(f"fill rate = {bs.fill_rate.mean:.1%}  95% CI [{bs.fill_rate.ci_low:.1%}, {bs.fill_rate.ci_high:.1%}]"
      f"  (MSE {bs.mse_fill_rate:.2e})")

## Inverse-transform sampling reproduces the empirical PMF
Cheat-sheet `x = F^{-1}(U)`.

In [ ]:
rng = np.random.default_rng(SEED)
epdf = decision[busiest]
draws = sample_from_epdf(epdf, 200_000, rng=rng)
sup = np.array(sorted(epdf)); tot = sum(epdf.values())
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(sup-0.18, [epdf[s]/tot for s in sup], width=0.36, label="empirical PMF", color="steelblue")
ax.bar(sup+0.18, [np.mean(draws==s) for s in sup], width=0.36, label="inverse-transform draws", color="orange")
ax.set_xlabel("rangeUp (ticks)"); ax.set_ylabel("probability"); ax.legend()
ax.set_title(f"Inverse-transform vs empirical PMF — cell {busiest}")
fig.tight_layout(); fig.savefig(FIGDIR/"mc_inverse_transform.png", dpi=120); plt.show()

## Parametric fitting + goodness-of-fit
Fit candidate families (geometric, neg-binomial, gamma, Weibull, half-normal), select by AIC, report KS. Overlay the best fit on the empirical histogram.

In [ ]:
fit = fit_distribution(epdf)
print(f"best family: {fit.family}  params={tuple(round(p,4) for p in fit.params)}  "
      f"AIC={fit.aic:.1f}  KS={fit.ks_stat:.3f}")
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(sup, [epdf[s]/tot for s in sup], width=0.6, alpha=0.6, label="empirical", color="steelblue")
xs = np.array(fit.support); ax.plot(xs, fit.pmf, "o-", ms=3, color="crimson", label=f"fit: {fit.family}")
ax.set_xlim(-0.5, sup.max()+1); ax.set_xlabel("rangeUp (ticks)"); ax.set_ylabel("probability"); ax.legend()
ax.set_title(f"Empirical vs fitted ({fit.family}, KS={fit.ks_stat:.3f}) — cell {busiest}")
fig.tight_layout(); fig.savefig(FIGDIR/"mc_fit_overlay.png", dpi=120); plt.show()

## Brownian-motion path model vs the reflection principle
Forward driftless BM (σ from the range level); rangeUp = running max. The simulated mean approaches the closed form E[max]=σ√(2T/π) as the path is refined.

In [ ]:
sig = sigma_by_cell[busiest]
ru, _ = simulate_gbm_ranges(sig, TAU, 20000, n_sub=200, round_ticks=False, rng=np.random.default_rng(SEED))
analytic = halfnormal_range_mean(sig, TAU)
print(f"sigma_bar={sig:.3f}  sim mean rangeUp={ru.mean():.2f}  reflection-principle={analytic:.2f}")
fig, ax = plt.subplots(figsize=(7,4))
ax.hist(ru, bins=50, density=True, alpha=0.6, color="seagreen", label="simulated rangeUp")
ax.axvline(ru.mean(), color="seagreen", ls="--", label=f"sim mean {ru.mean():.1f}")
ax.axvline(analytic, color="black", ls=":", label=f"E[max]=σ√(2T/π) {analytic:.1f}")
ax.set_xlabel("rangeUp (ticks)"); ax.set_ylabel("density"); ax.legend()
ax.set_title("Forward-GBM range vs reflection principle")
fig.tight_layout(); fig.savefig(FIGDIR/"mc_reflection_principle.png", dpi=120); plt.show()

## Three-way model comparison + σ stress test + validation vs backtest
Regime-marginal MC under empirical / fitted / gbm, against the historical v2 backtest. Fill rate is the robust quantity (slippage agreement is looser by construction).

In [ ]:
bt = run_backtest_rolling(ohlcv, tau=TAU, tick=tick, proper_days=days, side=SIDE,
                          fill_rate_target=TARGET, half_life=HALF_LIFE, M=M, N=N, K=K, j_start=J_START)
mc = {m: run_marginal_mc(decision, side=SIDE, fill_rate_target=TARGET, n_paths=40000,
                         range_model=m, tau=TAU, sigma_by_cell=sigma_by_cell,
                         cell_weights=cell_weights, fits=fits, seed=SEED)
      for m in ("empirical","fitted","gbm")}
print(f"backtest v2 fill={bt.fill_rate:.1%}")
for m,r in mc.items():
    print(f"  MC {m:9} fill={r.fill_rate.mean:.1%}  avg_slip={r.avg_slippage_ticks.mean:+.2f}t")
print("validation:", compare_mc_vs_backtest(mc["empirical"], bt, tol_fill=0.10)["flag"])

fig, ax = plt.subplots(figsize=(7,4))
names = list(mc) + ["backtest"]
fills = [mc[m].fill_rate.mean for m in mc] + [bt.fill_rate]
ax.bar(names, fills, color=["steelblue","crimson","seagreen","gray"])
ax.axhline(TARGET, ls="--", color="black", label=f"target {TARGET}")
ax.set_ylabel("fill rate"); ax.set_title("Fill rate: three MC models vs backtest"); ax.legend()
fig.tight_layout(); fig.savefig(FIGDIR/"mc_model_comparison.png", dpi=120); plt.show()

In [ ]:
scales = [0.5, 0.75, 1.0, 1.5, 2.0]
stress = stress_sigma(decision[busiest], side=SIDE, fill_rate_target=TARGET, n_paths=20000,
                      tau=TAU, sigma_bar=sigma_by_cell[busiest], scales=scales, seed=SEED)
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(scales, [s.fill_rate.mean for s in stress], "o-", color="purple")
ax.set_xlabel("σ scale"); ax.set_ylabel("fill rate"); ax.set_title(f"σ stress test — cell {busiest} (gbm)")
fig.tight_layout(); fig.savefig(FIGDIR/"mc_sigma_stress.png", dpi=120); plt.show()

## Phase C — Variance reduction
Antithetic, control-variate, conditional-MC and importance-sampling estimators; the ratio is plain-variance / reduced-variance (>1 means it helped).

In [ ]:
sig = sigma_by_cell[busiest]; e = decision[busiest]
vr = {
 "antithetic":  antithetic_slippage(e, side=SIDE, fill_rate_target=TARGET, n_paths=40000, tau=TAU, sigma_bar=sig, seed=SEED),
 "control":     control_variate_slippage(e, side=SIDE, fill_rate_target=TARGET, n_paths=40000, tau=TAU, sigma_bar=sig, seed=SEED),
 "conditional": conditional_mc_fill_rate(decision, cell_weights, fill_rate_target=TARGET),
 "importance":  importance_sampling_chase_tail(e, n_paths=40000, seed=SEED),
}
for k,v in vr.items():
    print(f"{k:12} variance-reduction ratio = {v.variance_reduction_ratio:6.1f}x")
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(list(vr), [v.variance_reduction_ratio for v in vr.values()], color="teal")
ax.axhline(1.0, ls="--", color="black"); ax.set_yscale("log")
ax.set_ylabel("variance-reduction ratio (log)"); ax.set_title("Variance reduction by technique")
fig.tight_layout(); fig.savefig(FIGDIR/"mc_variance_reduction.png", dpi=120); plt.show()